### **NMDC Parquet Discovery, Validation, and Safe Cleanup Prior to Delta Ingestion**

This cell performs a **controlled discovery, validation, and optional cleanup** workflow for Parquet files staged in object storage before they are ingested or registered as Delta tables in the Lakehouse.

#### **1. Environment and Spark Context Setup**

The cell begins by reading runtime environment variables and Spark configuration values required for:

* Identifying the **execution environment** (e.g., local, dev, or production)
* Ensuring the Spark session is correctly initialized for **Delta Lake operations**
* Confirming that required Spark features (e.g., Delta support) are enabled

This step ensures that downstream operations execute safely and consistently across environments.

In [1]:
spark = get_spark_session()
print(spark.conf.get("spark.sql.parquet.outputTimestampType"))

spark = get_spark_session(override={"spark.sql.parquet.outputTimestampType": "TIMESTAMP_MICROS"})
print(spark.conf.get("spark.sql.parquet.outputTimestampType"))


INT96
TIMESTAMP_MICROS


#### Recursive Parquet File Discovery**

Using the object storage client (e.g., MinIO / S3-compatible API), the cell:

* Recursively lists all objects under a specified bucket and prefix
* Filters for files with the `.parquet` extension
* Builds a list of candidate Parquet files for further inspection

This allows the pipeline to dynamically discover staged data without hardcoding file paths.

In [2]:
minio_client = get_minio_client()
#spark = get_spark_session()

In [3]:
spark.sql("show tables in nmdc_core").show(500, truncate=False)

+---------+------------------------------+-----------+
|namespace|tableName                     |isTemporary|
+---------+------------------------------+-----------+
|nmdc_core|annotation_terms_unified      |false      |
|nmdc_core|cog_categories                |false      |
|nmdc_core|cog_hierarchy_flat            |false      |
|nmdc_core|ec_hierarchy_flat             |false      |
|nmdc_core|ec_hierarchy_graph            |false      |
|nmdc_core|ec_terms                      |false      |
|nmdc_core|go_hierarchy_flat             |false      |
|nmdc_core|go_hierarchy_graph            |false      |
|nmdc_core|go_terms                      |false      |
|nmdc_core|kegg_ko_module                |false      |
|nmdc_core|kegg_ko_pathway               |false      |
|nmdc_core|kegg_ko_terms                 |false      |
|nmdc_core|kegg_module_terms             |false      |
|nmdc_core|kegg_pathway_terms            |false      |
|nmdc_core|metacyc_hierarchy_flat        |false      |
|nmdc_core

#### Recursive Parquet File Discovery**

Using the object storage client (e.g., MinIO / S3-compatible API), the cell:

* Recursively lists all objects under a specified bucket and prefix
* Filters for files with the `.parquet` extension
* Builds a list of candidate Parquet files for further inspection

This allows the pipeline to dynamically discover staged data without hardcoding file paths.

In [4]:
bucket = "cdm-lake"
prefix = "tenant-general-warehouse/nmdc/datasets/NMDC_gold/"
database = "nmdc_core"

In [5]:
parquet_objects = [
    obj.object_name
    for obj in minio_client.list_objects(
        bucket,
        prefix=prefix,
        recursive=True
    )
    if obj.object_name.endswith(".parquet")
]

print(f"Found {len(parquet_objects)} parquet files")

Found 65 parquet files


#### Helper Function for Safe Object Deletion**

A reusable helper function is defined to:

* Delete one or more objects from object storage
* Handle errors gracefully and surface meaningful exceptions
* Ensure partial deletes do not silently fail

This function is intentionally isolated so deletion logic is explicit and auditable.


In [8]:
from minio.deleteobjects import DeleteObject
from minio.error import S3Error

def delete_minio_prefix(minio_client, bucket, prefix):
    """
    Force-delete all objects under a MinIO prefix.
    """
    print(f" Deleting MinIO prefix: s3://{bucket}/{prefix}")

    objects = minio_client.list_objects(
        bucket,
        prefix=prefix,
        recursive=True
    )

    delete_list = (DeleteObject(obj.object_name) for obj in objects)

    errors = minio_client.remove_objects(bucket, delete_list)

    for err in errors:
        raise RuntimeError(f"Failed to delete object: {err}")

    print(" Prefix deleted successfully")

#### Iterative Table Validation Loop**

For each discovered Parquet file, the cell:

* Derives the intended **logical table name** from the file path
* Checks whether a corresponding table already exists in the Spark catalog
* Logs the table name and existence status

This prevents accidental overwrites and enforces idempotent ingestion behavior.


#### Conditional Cleanup of Stale or Invalid Tables**

If a table is detected but deemed invalid or stale:

* The existing table is explicitly dropped from the catalog
* The associated Parquet files are deleted from object storage using the helper function
* Errors during deletion or table removal are surfaced immediately

This ensures the Lakehouse does not accumulate orphaned metadata or inconsistent storage state.


#### Defensive Error Handling**

The cell wraps all destructive operations in `try / except` blocks to:

* Prevent cascading failures
* Provide actionable error messages
* Maintain control over cleanup behavior in production environments


In [9]:
overwrite = False
force_cleanup = True

In [12]:
import os
from pyspark.errors.exceptions.connect import SparkRuntimeException
from pyspark.sql.functions import countDistinct

for obj in parquet_objects:
    parquet_path = f"s3a://{bucket}/{obj}"
    table_name = os.path.splitext(os.path.basename(obj))[0]
    full_table_name = f"{database}.{table_name}"

    stat = minio_client.stat_object(bucket, obj)
    size_bytes = stat.size
    size_gb = size_bytes / (1024**3)

    try:
        table_exists = spark.catalog.tableExists(full_table_name)

        print("=" * 80)
        print(f"Processing {parquet_path}")
        print(f"Target table: {full_table_name}")
        print(f"Source file size: {size_gb:.2f} GB")

        if not table_exists:
            print(f"{full_table_name} does not exist (overwrite=False)")
            print(f"Dropping table metadata if present: {full_table_name}")

            spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")

            df = spark.read.parquet(parquet_path)

            try:
                if full_table_name == "nmdc_core.covstats_gold":
                    df.printSchema()
                    if "workflow_id" not in df.columns:
                        raise ValueError("Partition column 'workflow_id' not found")

                    (
                        df.write
                        .format("delta")
                        .mode("overwrite")
                        .partitionBy("workflow_id")
                        .saveAsTable(full_table_name)
                    )
                else:
                    (
                        df.write
                        .format("delta")
                        .mode("overwrite")
                        .saveAsTable(full_table_name)
                    )

            except SparkRuntimeException as e:
                msg = str(e)
                if "LOCATION_ALREADY_EXISTS" not in msg:
                    raise

                print(f"Caught LOCATION_ALREADY_EXISTS for {full_table_name}")

                if force_cleanup:
                    err_bucket = "cdm-lake"
                    err_prefix = f"tenant-sql-warehouse/nmdc/{database}.db/{table_name}"

                    delete_minio_prefix(minio_client, err_bucket, err_prefix)

                    print(f"Retrying write after cleanup: {full_table_name}")

                    if full_table_name == "nmdc_core.covstats_gold":
                        (
                            df.write
                            .format("delta")
                            .mode("overwrite")
                            .partitionBy("workflow_id")
                            .saveAsTable(full_table_name)
                        )
                    else:
                        (
                            df.write
                            .format("delta")
                            .mode("overwrite")
                            .saveAsTable(full_table_name)
                        )

        # TABLE EXISTS
        else:
            if overwrite:
                print(f"Overwriting existing table {full_table_name}")

                df = spark.read.parquet(parquet_path)

                (
                    df.write
                    .format("delta")
                    .mode("overwrite")
                    .insertInto(full_table_name)
                )
            else:
                print(f"Skipping {full_table_name} (already exists, overwrite=False)")

        print(f"Completed {full_table_name}")

    except Exception as e:
        print(f"Failed for {parquet_path}: {e}")


Processing s3a://cdm-lake/tenant-general-warehouse/nmdc/datasets/NMDC_gold/annotation_hierarchies/annotation_crossrefs.parquet
Target table: nmdc_core.annotation_crossrefs
Source file size: 0.00 GB
Skipping nmdc_core.annotation_crossrefs (already exists, overwrite=False)
Completed nmdc_core.annotation_crossrefs
Processing s3a://cdm-lake/tenant-general-warehouse/nmdc/datasets/NMDC_gold/annotation_hierarchies/annotation_hierarchies_unified.parquet
Target table: nmdc_core.annotation_hierarchies_unified
Source file size: 0.00 GB
Skipping nmdc_core.annotation_hierarchies_unified (already exists, overwrite=False)
Completed nmdc_core.annotation_hierarchies_unified
Processing s3a://cdm-lake/tenant-general-warehouse/nmdc/datasets/NMDC_gold/annotation_hierarchies/annotation_terms_unified.parquet
Target table: nmdc_core.annotation_terms_unified
Source file size: 0.00 GB
Skipping nmdc_core.annotation_terms_unified (already exists, overwrite=False)
Completed nmdc_core.annotation_terms_unified
Proce

In [ ]:
s3a://cdm-lake/tenant-general-warehouse/nmdc/datasets/NMDC_gold/covstats_gold.parquet

In [6]:
target_files = 200  # choose based on size (see below)
covstats_gold_path = "s3a://cdm-lake/tenant-general-warehouse/nmdc/datasets/NMDC_gold/covstats_gold.parquet"
covstats_gold_staging = "s3a://cdm-lake/tenant-general-warehouse/nmdc/datasets/NMDC_gold/covstats_gold_staging/"

c_df = spark.read.parquet(covstats_gold_path)

In [8]:
c_df.printSchema()

root
 |-- file_id: string (nullable = true)
 |-- file_name: string (nullable = true)
 |-- feature_id: string (nullable = true)
 |-- Avg_fold: float (nullable = true)
 |-- Covered_percent: float (nullable = true)
 |-- Length: integer (nullable = true)
 |-- Median_fold: float (nullable = true)
 |-- RPK: float (nullable = true)
 |-- RPKM: float (nullable = true)
 |-- Ref_GC: float (nullable = true)
 |-- Std_Dev: float (nullable = true)
 |-- TPM: float (nullable = true)
 |-- contig_id: string (nullable = true)
 |-- normalized_contig_id: string (nullable = true)
 |-- reads: integer (nullable = true)
 |-- rel_abundance: float (nullable = true)
 |-- workflow_id: string (nullable = true)

